# 基于平衡二叉树结构的并行前缀和实验

前缀和是并行计算中的经典问题，常见于数据压缩、并行排序、稀疏计算和并行算法中的索引构造。本实验围绕前缀和计算展开，完成平衡二叉树遍历、数据分块、Host侧与Device侧协同执行以及正确性验证。对于给定输入序列$x_0,x_1,\ldots,x_{n-1}$，实验在AscendC环境中采用静态Tensor编程方式实现并行前缀和：

$$
y_i=\sum_{j=0}^{i}x_j,\quad 0\le i<n
$$

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建实验目录并加载CANN环境；
3. 问题分析：分析输入输出、计算公式、数据布局、分块策略、平衡二叉树遍历方法和实验参数设置；
4. 核函数开发：实现平衡二叉树遍历和块间前缀和累加；
5. 结果验证与性能分析：准备Host侧输入数据和CPU参考结果，完成工程构建、算子运行、结果验证和性能分析；
6. 实验总结：归纳平衡二叉树遍历、数据分块、块间前缀和累加和多核并行在并行前缀和中的实现过程。

---
## 1. 实验概述

本实验基于平衡二叉树数据结构，在Ascend C环境中采用静态Tensor编程方式实现并行前缀和。实验将输入数据划分为若干连续数据块，在各数据块内部利用平衡树方法计算局部前缀和，并对各数据块进行块间前缀和累加，最终得到全局前缀和。通过本实验，可以理解平衡二叉树结构在并行计算中的应用，掌握平衡树方法与划分法，并理解通过构造树形结构提高并行度的算法设计思想。

### 1.1 实验目标

完成本实验后应达到以下目标：

1. 掌握树形数据结构的组织方式。能够基于平衡二叉树结构组织块内前缀和计算，理解将输入数据映射为叶子节点、将子树局部和保存为内部节点信息的表示方法，并理解通过构造树形结构实现并行计算的算法设计思想。
2. 理解平衡树方法与划分法。能够理解平衡树方法中正向遍历和反向遍历的作用，理解划分法思想与静态Tensor编程方式，将输入数据划分为连续数据块，并完成全局内存与局部内存之间的数据搬运、片上计算和结果写回。
3. 具备正确性验证和性能分析能力。能够通过最大绝对误差和最大相对误差指标，对昇腾NPU计算结果与CPU计算结果进行正确性验证；能够根据块内扫描时间、Host侧偏移量计算时间、块间前缀和累加时间及整体执行时间，分析NPU并行计算性能。

### 1.2 前置知识

本实验要求提前具备以下基础：

1. 前缀和计算基础：理解前缀和的数学含义和累加过程，明确第i个输出元素由输入序列前i个元素累加得到。能够从循环计算的角度理解串行前缀和，并认识到前缀和的主要特点是元素之间存在顺序依赖。
2. 平衡树方法基础：理解利用平衡树方法组织并行计算依赖的二叉计算树。理解正向遍历用于生成子树局部和信息，反向遍历用于传播前缀信息；认识树高、叶子数量和数据块长度之间的关系。
3. 数据分块思想基础：理解分块计算的基本思想，即把大规模输入序列划分为较小的数据块，使设备侧计算能够围绕块进行组织。进一步理解块内计算和块间偏移量之间的关系，认识到块间辅助信息是连接局部结果和全局结果的关键。
4. Ascend C开发基础：Host侧负责输入数据准备、运行时管理、Device侧内存申请、Kernel启动和结果校验；Device侧执行Kernel核函数，完成前缀和的核心计算。实验前应熟悉工程编译与脚本运行方法，同时掌握性能指标查看方法。
5. 静态Tensor编程基础：理解静态Tensor编程方式要求开发者显式规划数据块边界、LocalTensor地址和存储位置，并根据数据搬入、计算和写回之间的依赖关系管理同步。理解数据通常先从全局内存搬入局部内存，再在片上完成计算，最后写回全局内存。

### 1.3 实验要点

实验中应重点关注以下内容：

1. 数据组织：按照连续内存方式保存输入数组、辅助数组和输出数组，确保Host侧和Device侧对数组规模、块划分方式和边界范围保持一致；
2. 数据搬运：将当前计算所需的数据片段从全局内存搬入局部内存，在LocalTensor中使用平衡二叉树计算局部前缀和；
3. 平衡树方法实现：在Device侧，利用平衡树方法在AI Core上对每个数据分块计算局部前缀和，将输入元素作为叶子节点构建一棵平衡二叉树,然后通过自叶向根的往返遍历，实现并行前缀和；
4. 结果验证与性能分析：根据CPU串行计算结果对NPU并行计算结果进行正确性验证；并根据二者的总执行时间计算加速比，分析NPU并行实现相对CPU串行实现的性能提升。

---
## 2. 环境准备

### 2.1 创建实验目录并加载CANN环境

本小节创建实验所需目录，并加载Ascend CANN环境变量。

目录划分如下：

- `src/02.00_intra_prefix_sum_balanced_tree/include`：保存Host侧公共头文件和实验参数定义。
- `src/02.00_intra_prefix_sum_balanced_tree/src`：保存CPU参考实现、平衡二叉树模拟实现和主程序。
- `src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel`：保存Device侧核函数代码，包括块内扫描和块间偏移回加。
- `src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/host_launch`：保存Host侧核函数调用代码。
- `src/02.00_intra_prefix_sum_balanced_tree/scripts`：保存构建与运行脚本。
- `src/02.00_intra_prefix_sum_balanced_tree/results`：保存实验结果。

如果当前环境已安装CANN，下面的代码会加载`set_env.sh`。如果没有找到该文件，仍然可以继续查看和生成代码，但完整编译和运行需要在已配置CANN的环境中完成。

In [ ]:
from pathlib import Path
import os
import subprocess

WORK_DIR = Path("src/02.00_intra_prefix_sum_balanced_tree").resolve()
SRC_DIR = WORK_DIR / "src"

# 创建本实验目录结构
(WORK_DIR / "results").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "scripts").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "include").mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "ascend_ops" / "op_kernel").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "ascend_ops" / "host_launch").mkdir(parents=True, exist_ok=True)

# 自动查找并加载CANN环境
arch = os.uname().machine
candidate_paths = []

for name in ["ASCEND_INSTALL_PATH", "ASCEND_TOOLKIT_HOME"]:
    value = os.environ.get(name)
    if value:
        candidate_paths.append(Path(value))

ascend_home = os.environ.get("ASCEND_HOME_PATH")
if ascend_home:
    candidate_paths.append(Path(ascend_home) / f"{arch}-linux")

candidate_paths.extend(
    sorted(Path("/opt/conda/Ascend").glob(f"cann-*/{arch}-linux"), reverse=True)
)
candidate_paths.append(Path("/usr/local/Ascend/ascend-toolkit/latest"))
candidate_paths.extend(
    sorted(Path("/usr/local/Ascend/ascend-toolkit").glob(f"*/{arch}-linux"), reverse=True)
)

set_env = None
for item in candidate_paths:
    if (item / "set_env.sh").exists():
        set_env = item / "set_env.sh"
        break

if set_env is not None:
    env = subprocess.check_output(
        f"bash -l -c 'source {set_env} && env'",
        shell=True,
        text=True,
    )
    for line in env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_INSTALL_PATH"] = str(set_env.parent)
    print("Ascend environment loaded from:", set_env)
else:
    print("Ascend set_env.sh was not found. Build and run need CANN.")

print("Experiment directory:", WORK_DIR)
print("Source directory:", SRC_DIR)


### 2.2 写入工程公共文件

本节写入公共头文件、CPU参考实现和Host侧启动代码。Device侧核函数将在第4节写入，CMake构建文件和运行脚本将在第5节写入。运行到第5节后，`src/02.00_intra_prefix_sum_balanced_tree`目录下将形成可直接编译执行的AscendC工程。

In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/include/prefix_sum_common.h
#pragma once

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <stdexcept>
#include <string>
#include <vector>

namespace psum {

struct PrefixSumConfig {
    uint64_t n = 1ull << 20;
    uint32_t block_len = 1024;
    uint32_t block_dim = 1;
    uint32_t seed = 1234;
    bool sweep = false;
};

struct Metrics {
    double max_abs_error = 0.0;
    double max_rel_error = 0.0;
};

struct TimingUs {
    double block_scan_us = 0.0;
    double host_offset_us = 0.0;
    double add_offset_us = 0.0;
    double total_us = 0.0;
};

struct TwoStageResult {
    std::vector<float> y;
    std::vector<float> block_sum;
    std::vector<float> block_offset;
    TimingUs timing;
};

inline void check_config(uint64_t n, uint32_t block_len, uint32_t dtype_size = sizeof(float)) {
    if (block_len == 0) {
        throw std::invalid_argument("block_len must be > 0");
    }
    if (n == 0) {
        throw std::invalid_argument("n must be > 0");
    }
    if (n % block_len != 0) {
        throw std::invalid_argument("n must be divisible by block_len");
    }
    if ((static_cast<uint64_t>(block_len) * dtype_size) % 32 != 0) {
        throw std::invalid_argument("block_len * sizeof(T) must be a multiple of 32 bytes");
    }
}

class Timer {
public:
    Timer() : start_(std::chrono::high_resolution_clock::now()) {}
    double elapsed_us() const {
        auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(end - start_).count();
    }
private:
    std::chrono::high_resolution_clock::time_point start_;
};

inline double estimate_bytes(uint64_t n, uint64_t num_blocks) {
    // Kernel1: read x + write y + write block_sum.
    // Kernel2: read y + read block_offset + write y.
    return static_cast<double>(sizeof(float)) * (4.0 * static_cast<double>(n) + 2.0 * static_cast<double>(num_blocks));
}

inline void print_result_row(uint64_t n,
                             uint32_t block_len,
                             uint64_t num_blocks,
                             const TimingUs& t,
                             const Metrics& m) {
    const double bytes = estimate_bytes(n, num_blocks);
    const double gbps = bytes / (t.total_us * 1e-6) / 1e9;
    std::cout << std::setw(10) << n
              << std::setw(12) << block_len
              << std::setw(12) << num_blocks
              << std::setw(14) << std::fixed << std::setprecision(2) << t.block_scan_us
              << std::setw(14) << t.host_offset_us
              << std::setw(14) << t.add_offset_us
              << std::setw(14) << t.total_us
              << std::setw(12) << std::setprecision(3) << gbps
              << std::setw(14) << std::scientific << m.max_abs_error
              << std::setw(14) << m.max_rel_error
              << std::defaultfloat
              << "\n";
}

} // namespace psum


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/include/prefix_sum_cpu.h
#pragma once

#include "prefix_sum_common.h"

namespace psum {

std::vector<float> make_input(uint64_t n, uint32_t seed);
std::vector<float> inclusive_scan_reference(const std::vector<float>& x);
Metrics compare_vectors(const std::vector<float>& got, const std::vector<float>& ref);
void dump_sample(const std::vector<float>& x, const std::vector<float>& y, const std::vector<float>& ref, size_t count = 8);

} // namespace psum


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/include/prefix_sum_balanced_tree_cpu.h
#pragma once

#include "prefix_sum_cpu.h"

#include <cstdint>
#include <vector>

namespace psum {

constexpr uint32_t kBalancedTreeBlockLen = 1024;
constexpr uint32_t kBalancedTreeLevels = 10;
constexpr uint32_t kBalancedTreeNodeCount = 2 * kBalancedTreeBlockLen - 1;

TwoStageResult two_stage_prefix_sum_balanced_tree_cpu(const std::vector<float>& x);

} // namespace psum


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/src/prefix_sum_cpu.cpp
#include "prefix_sum_cpu.h"

#include <random>

namespace psum {

std::vector<float> make_input(uint64_t n, uint32_t seed) {
    std::vector<float> x(n);
    std::mt19937 rng(seed);
    // Use small positive values to keep floating-point accumulation stable enough
    // for a classroom reference check while still avoiding a constant pattern.
    std::uniform_real_distribution<float> dist(0.0001f, 0.01f);
    for (auto& v : x) {
        v = dist(rng);
    }
    return x;
}

std::vector<float> inclusive_scan_reference(const std::vector<float>& x) {
    std::vector<float> y(x.size());
    float acc = 0.0f;
    for (size_t i = 0; i < x.size(); ++i) {
        acc += x[i];
        y[i] = acc;
    }
    return y;
}

Metrics compare_vectors(const std::vector<float>& got, const std::vector<float>& ref) {
    if (got.size() != ref.size()) {
        throw std::invalid_argument("compare_vectors: size mismatch");
    }
    Metrics m;
    for (size_t i = 0; i < got.size(); ++i) {
        const double abs_err = std::abs(static_cast<double>(got[i]) - static_cast<double>(ref[i]));
        const double denom = std::max(1e-12, std::abs(static_cast<double>(ref[i])));
        const double rel_err = abs_err / denom;
        m.max_abs_error = std::max(m.max_abs_error, abs_err);
        m.max_rel_error = std::max(m.max_rel_error, rel_err);
    }
    return m;
}

void dump_sample(const std::vector<float>& x, const std::vector<float>& y, const std::vector<float>& ref, size_t count) {
    const size_t n = std::min({x.size(), y.size(), ref.size(), count});
    std::cout << "sample(first " << n << "):\n";
    for (size_t i = 0; i < n; ++i) {
        std::cout << "  i=" << i << " x=" << x[i] << " y=" << y[i] << " ref=" << ref[i] << "\n";
    }
}

} // namespace psum


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/src/prefix_sum_balanced_tree_cpu.cpp
#include "prefix_sum_balanced_tree_cpu.h"

#include <array>

namespace psum {

namespace {

std::array<uint32_t, kBalancedTreeLevels + 1> make_level_offsets() {
    std::array<uint32_t, kBalancedTreeLevels + 1> offsets{};
    offsets[0] = 0;
    for (uint32_t h = 1; h <= kBalancedTreeLevels; ++h) {
        offsets[h] = offsets[h - 1] + (kBalancedTreeBlockLen >> (h - 1));
    }
    return offsets;
}

void block_scan_balanced_tree(const float* xBlock,
                              float* yBlock,
                              float& blockSum,
                              const std::array<uint32_t, kBalancedTreeLevels + 1>& levelOffset) {
    std::vector<float> bTree(kBalancedTreeNodeCount, 0.0f);
    std::vector<float> cTree(kBalancedTreeNodeCount, 0.0f);

    for (uint32_t i = 0; i < kBalancedTreeBlockLen; ++i) {
        bTree[levelOffset[0] + i] = xBlock[i];
    }

    for (uint32_t h = 1; h <= kBalancedTreeLevels; ++h) {
        const uint32_t curOffset = levelOffset[h];
        const uint32_t prevOffset = levelOffset[h - 1];
        const uint32_t nodeCount = kBalancedTreeBlockLen >> h;
        for (uint32_t j = 0; j < nodeCount; ++j) {
            bTree[curOffset + j] = bTree[prevOffset + 2 * j] + bTree[prevOffset + 2 * j + 1];
        }
    }

    const uint32_t rootOffset = levelOffset[kBalancedTreeLevels];
    cTree[rootOffset] = bTree[rootOffset];

    for (int h = static_cast<int>(kBalancedTreeLevels) - 1; h >= 0; --h) {
        const uint32_t curOffset = levelOffset[static_cast<uint32_t>(h)];
        const uint32_t parentOffset = levelOffset[static_cast<uint32_t>(h) + 1];
        const uint32_t nodeCount = kBalancedTreeBlockLen >> static_cast<uint32_t>(h);

        for (uint32_t j0 = 0; j0 < nodeCount; ++j0) {
            const uint32_t j = j0 + 1; // Chen Guoliang textbook notation is 1-based.
            if (j == 1) {
                cTree[curOffset + j0] = bTree[curOffset + j0];
            } else if ((j & 1u) == 0u) {
                cTree[curOffset + j0] = cTree[parentOffset + (j / 2u - 1u)];
            } else {
                cTree[curOffset + j0] = cTree[parentOffset + ((j - 1u) / 2u - 1u)] +
                                        bTree[curOffset + j0];
            }
        }
    }

    for (uint32_t i = 0; i < kBalancedTreeBlockLen; ++i) {
        yBlock[i] = cTree[levelOffset[0] + i];
    }
    blockSum = bTree[rootOffset];
}

} // namespace

TwoStageResult two_stage_prefix_sum_balanced_tree_cpu(const std::vector<float>& x) {
    check_config(x.size(), kBalancedTreeBlockLen, sizeof(float));
    const uint64_t n = x.size();
    const uint64_t numBlocks = n / kBalancedTreeBlockLen;
    const auto levelOffset = make_level_offsets();

    TwoStageResult result;
    result.y.assign(n, 0.0f);
    result.block_sum.assign(numBlocks, 0.0f);
    result.block_offset.assign(numBlocks, 0.0f);

    Timer totalTimer;

    {
        Timer timer;
        for (uint64_t b = 0; b < numBlocks; ++b) {
            const uint64_t base = b * kBalancedTreeBlockLen;
            block_scan_balanced_tree(&x[base],
                                     &result.y[base],
                                     result.block_sum[b],
                                     levelOffset);
        }
        result.timing.block_scan_us = timer.elapsed_us();
    }

    {
        Timer timer;
        float offset = 0.0f;
        for (uint64_t b = 0; b < numBlocks; ++b) {
            result.block_offset[b] = offset;
            offset += result.block_sum[b];
        }
        result.timing.host_offset_us = timer.elapsed_us();
    }

    {
        Timer timer;
        for (uint64_t b = 0; b < numBlocks; ++b) {
            const uint64_t base = b * kBalancedTreeBlockLen;
            const float offset = result.block_offset[b];
            for (uint32_t j = 0; j < kBalancedTreeBlockLen; ++j) {
                result.y[base + j] += offset;
            }
        }
        result.timing.add_offset_us = timer.elapsed_us();
    }

    result.timing.total_us = totalTimer.elapsed_us();
    return result;
}

} // namespace psum


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/src/main_balanced_tree.cpp
#include "prefix_sum_balanced_tree_cpu.h"

#include <cstring>
#include <utility>

namespace {

struct Config {
    uint64_t n = 1ull << 20;
    uint32_t seed = 1234;
    uint32_t warmup = 5;
    uint32_t repeat = 30;
    bool printOutput = false;
};

void usage(const char* argv0) {
    std::cout << "Usage: " << argv0
              << " [--n N] [--seed S] [--warmup W] [--repeat R] [--print-output]\n"
              << "Balanced-tree blockLen is fixed at " << psum::kBalancedTreeBlockLen << ".\n"
              << "Example:\n"
              << "  " << argv0 << " --n 1048576 --warmup 5 --repeat 30\n";
}

Config parse_args(int argc, char** argv) {
    Config cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) {
                throw std::invalid_argument("missing value after " + name);
            }
            return argv[++i];
        };
        if (arg == "--n") {
            cfg.n = std::stoull(need_value(arg));
        } else if (arg == "--seed") {
            cfg.seed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(std::stoul(need_value(arg)));
            if (cfg.repeat == 0) {
                throw std::invalid_argument("--repeat must be positive");
            }
        } else if (arg == "--print-output") {
            cfg.printOutput = true;
        } else if (arg == "--block-len") {
            const uint32_t blockLen = static_cast<uint32_t>(std::stoul(need_value(arg)));
            if (blockLen != psum::kBalancedTreeBlockLen) {
                throw std::invalid_argument("balanced-tree demo fixes --block-len at 1024");
            }
        } else if (arg == "--help" || arg == "-h") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return cfg;
}

} // namespace

int main(int argc, char** argv) {
    try {
        const auto cfg = parse_args(argc, argv);
        psum::check_config(cfg.n, psum::kBalancedTreeBlockLen);

        auto x = psum::make_input(cfg.n, cfg.seed);
        auto ref = psum::inclusive_scan_reference(x);

        for (uint32_t i = 0; i < cfg.warmup; ++i) {
            (void)psum::two_stage_prefix_sum_balanced_tree_cpu(x);
        }

        psum::TimingUs timingSum;
        psum::TwoStageResult result;
        for (uint32_t i = 0; i < cfg.repeat; ++i) {
            auto current = psum::two_stage_prefix_sum_balanced_tree_cpu(x);
            timingSum.block_scan_us += current.timing.block_scan_us;
            timingSum.host_offset_us += current.timing.host_offset_us;
            timingSum.add_offset_us += current.timing.add_offset_us;
            timingSum.total_us += current.timing.total_us;
            result = std::move(current);
        }
        const double repeat = static_cast<double>(cfg.repeat);
        result.timing.block_scan_us = timingSum.block_scan_us / repeat;
        result.timing.host_offset_us = timingSum.host_offset_us / repeat;
        result.timing.add_offset_us = timingSum.add_offset_us / repeat;
        result.timing.total_us = timingSum.total_us / repeat;
        auto metrics = psum::compare_vectors(result.y, ref);

        std::cout << std::setw(10) << "N"
                  << std::setw(12) << "blockLen"
                  << std::setw(12) << "numBlocks"
                  << std::setw(14) << "scan_us"
                  << std::setw(14) << "offset_us"
                  << std::setw(14) << "add_us"
                  << std::setw(14) << "total_us"
                  << std::setw(12) << "GB/s"
                  << std::setw(14) << "max_abs"
                  << std::setw(14) << "max_rel"
                  << "\n";
        psum::print_result_row(cfg.n,
                               psum::kBalancedTreeBlockLen,
                               cfg.n / psum::kBalancedTreeBlockLen,
                               result.timing,
                               metrics);
        std::cout << "  version=two_stage_balanced_tree_cpu_simulator"
                  << ", blockLen=" << psum::kBalancedTreeBlockLen
                  << ", warmup=" << cfg.warmup
                  << ", repeat=" << cfg.repeat
                  << "\n";

        if (cfg.printOutput) {
            psum::dump_sample(x, result.y, ref, 8);
        }
        return 0;
    } catch (const std::exception& e) {
        std::cerr << "error: " << e.what() << "\n";
        usage(argv[0]);
        return 1;
    }
}


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/host_launch/prefix_sum_npu_main_balanced_tree.cpp
#include <acl/acl.h>
#include <aclrtlaunch_add_block_offset.h>
#include <aclrtlaunch_block_scan_stage1_balanced_tree.h>

#include "prefix_sum_balanced_tree_cpu.h"

#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <limits>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                                 \
    do {                                                                                \
        aclError _ret = (expr);                                                        \
        if (_ret != ACL_SUCCESS) {                                                     \
            throw std::runtime_error(std::string("ACL error: ") + #expr +             \
                                     ", code=" + std::to_string(static_cast<int>(_ret))); \
        }                                                                               \
    } while (0)

struct Config {
    int32_t device = 0;
    uint64_t n = 1ull << 20;
    uint32_t blockDim = 32;
    uint32_t warmup = 5;
    uint32_t repeat = 30;
    uint32_t seed = 1234;
    bool printOutput = false;
};

static void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "Options:\n"
              << "  --device <id>       device id, default: 0\n"
              << "  --n <num>           total element count, default: 1048576\n"
              << "  --block-len <num>   accepted only when num is 1024\n"
              << "  --block-dim <num>   AI Core launch blockDim, default: 32\n"
              << "  --warmup <num>      warmup count, default: 5\n"
              << "  --repeat <num>      repeat count, default: 30\n"
              << "  --seed <num>        random seed, default: 1234\n"
              << "  --print-output      print first 8 output values\n"
              << "  -h, --help          show help\n"
              << "Balanced-tree blockLen is fixed at " << psum::kBalancedTreeBlockLen << ".\n";
}

static Config parse_args(int argc, char** argv) {
    Config cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + name);
            return argv[++i];
        };
        if (arg == "--device") {
            cfg.device = std::stoi(need_value(arg));
        } else if (arg == "--n") {
            cfg.n = std::stoull(need_value(arg));
        } else if (arg == "--block-len") {
            const uint32_t blockLen = static_cast<uint32_t>(std::stoul(need_value(arg)));
            if (blockLen != psum::kBalancedTreeBlockLen) {
                throw std::invalid_argument("balanced-tree demo fixes --block-len at 1024");
            }
        } else if (arg == "--block-dim") {
            cfg.blockDim = static_cast<uint32_t>(std::stoul(need_value(arg)));
            if (cfg.blockDim == 0) {
                throw std::invalid_argument("--block-dim must be positive");
            }
        } else if (arg == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(std::stoul(need_value(arg)));
            if (cfg.repeat == 0) {
                throw std::invalid_argument("--repeat must be positive");
            }
        } else if (arg == "--seed") {
            cfg.seed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--print-output") {
            cfg.printOutput = true;
        } else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return cfg;
}

static void exclusive_scan_host(const std::vector<float>& blockSum,
                                std::vector<float>& blockOffset) {
    blockOffset.resize(blockSum.size());
    float acc = 0.0f;
    for (size_t i = 0; i < blockSum.size(); ++i) {
        blockOffset[i] = acc;
        acc += blockSum[i];
    }
}

static void run_two_stage_balanced_tree(const Config& cfg) {
    constexpr uint32_t blockLen = psum::kBalancedTreeBlockLen;
    psum::check_config(cfg.n, blockLen, sizeof(float));
    if (cfg.n > std::numeric_limits<uint32_t>::max()) {
        throw std::invalid_argument("--n exceeds uint32_t kernel argument range");
    }
    const uint32_t numBlocks = static_cast<uint32_t>(cfg.n / blockLen);
    const size_t dataBytes = static_cast<size_t>(cfg.n) * sizeof(float);
    const size_t blockBytes = static_cast<size_t>(numBlocks) * sizeof(float);

    std::vector<float> x = psum::make_input(cfg.n, cfg.seed);
    std::vector<float> ref = psum::inclusive_scan_reference(x);
    std::vector<float> y(cfg.n, 0.0f);
    std::vector<float> blockSum(numBlocks, 0.0f);
    std::vector<float> blockOffset(numBlocks, 0.0f);

    void* xDevice = nullptr;
    void* yDevice = nullptr;
    void* blockSumDevice = nullptr;
    void* blockOffsetDevice = nullptr;
    aclrtStream stream = nullptr;

    ACL_CHECK(aclrtCreateStream(&stream));
    ACL_CHECK(aclrtMalloc(&xDevice, dataBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&yDevice, dataBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&blockSumDevice, blockBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&blockOffsetDevice, blockBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMemcpy(xDevice, dataBytes, x.data(), dataBytes, ACL_MEMCPY_HOST_TO_DEVICE));

    auto run_once = [&](bool timed, psum::TimingUs* timing) {
        psum::Timer totalTimer;
        {
            psum::Timer timer;
            ACL_CHECK(aclrtMemset(yDevice, dataBytes, 0, dataBytes));
            ACL_CHECK(aclrtMemset(blockSumDevice, blockBytes, 0, blockBytes));
            ACLRT_LAUNCH_KERNEL(block_scan_stage1_balanced_tree)(cfg.blockDim, stream,
                                                                 xDevice, yDevice, blockSumDevice,
                                                                 static_cast<uint32_t>(cfg.n),
                                                                 numBlocks, blockLen, cfg.blockDim);
            ACL_CHECK(aclrtSynchronizeStream(stream));
            if (timed) timing->block_scan_us += timer.elapsed_us();
        }
        {
            psum::Timer timer;
            ACL_CHECK(aclrtMemcpy(blockSum.data(), blockBytes, blockSumDevice, blockBytes,
                                  ACL_MEMCPY_DEVICE_TO_HOST));
            exclusive_scan_host(blockSum, blockOffset);
            ACL_CHECK(aclrtMemcpy(blockOffsetDevice, blockBytes, blockOffset.data(), blockBytes,
                                  ACL_MEMCPY_HOST_TO_DEVICE));
            if (timed) timing->host_offset_us += timer.elapsed_us();
        }
        {
            psum::Timer timer;
            ACLRT_LAUNCH_KERNEL(add_block_offset)(cfg.blockDim, stream,
                                                  yDevice, blockOffsetDevice,
                                                  static_cast<uint32_t>(cfg.n),
                                                  numBlocks, blockLen, cfg.blockDim);
            ACL_CHECK(aclrtSynchronizeStream(stream));
            if (timed) timing->add_offset_us += timer.elapsed_us();
        }
        if (timed) timing->total_us += totalTimer.elapsed_us();
    };

    psum::TimingUs timing;
    for (uint32_t i = 0; i < cfg.warmup; ++i) run_once(false, &timing);
    for (uint32_t i = 0; i < cfg.repeat; ++i) run_once(true, &timing);
    if (cfg.repeat > 0) {
        timing.block_scan_us /= cfg.repeat;
        timing.host_offset_us /= cfg.repeat;
        timing.add_offset_us /= cfg.repeat;
        timing.total_us /= cfg.repeat;
    }

    ACL_CHECK(aclrtMemcpy(y.data(), dataBytes, yDevice, dataBytes, ACL_MEMCPY_DEVICE_TO_HOST));
    auto metrics = psum::compare_vectors(y, ref);
    psum::print_result_row(cfg.n, blockLen, numBlocks, timing, metrics);
    std::cout << "  version=two_stage_multicore_balanced_tree_static_tensor"
              << ", launchBlockDim=" << cfg.blockDim
              << ", warmup=" << cfg.warmup
              << ", repeat=" << cfg.repeat << "\n";
    if (cfg.printOutput) psum::dump_sample(x, y, ref, 8);

    aclrtFree(blockOffsetDevice);
    aclrtFree(blockSumDevice);
    aclrtFree(yDevice);
    aclrtFree(xDevice);
    aclrtDestroyStream(stream);
}

int main(int argc, char** argv) {
    try {
        const Config cfg = parse_args(argc, argv);
        ACL_CHECK(aclInit(nullptr));
        ACL_CHECK(aclrtSetDevice(cfg.device));

        std::cout << std::setw(10) << "N"
                  << std::setw(12) << "blockLen"
                  << std::setw(12) << "numBlocks"
                  << std::setw(14) << "scan_us"
                  << std::setw(14) << "offset_us"
                  << std::setw(14) << "add_us"
                  << std::setw(14) << "total_us"
                  << std::setw(12) << "GB/s"
                  << std::setw(14) << "max_abs"
                  << std::setw(14) << "max_rel"
                  << "\n";

        run_two_stage_balanced_tree(cfg);

        ACL_CHECK(aclrtResetDevice(cfg.device));
        ACL_CHECK(aclFinalize());
        return 0;
    } catch (const std::exception& e) {
        std::cerr << "error: " << e.what() << "\n";
        usage(argv[0]);
        return 1;
    }
}


---
## 3. 问题分析

本节分析前缀和算子的输入输出、计算公式、数据布局、分块策略、平衡二叉树遍历方法和实验参数设置。后续核函数开发将围绕本节内容展开。

### 3.1 输入输出与计算公式

本实验实现一维浮点数组的包含式前缀和计算。输入为长度为$n$的一维数组：

$$
x = [x_0,x_1,\ldots,x_{n-1}]
$$

输出为同样长度的一维数组：

$$
y = [y_0,y_1,\ldots,y_{n-1}]
$$

其中，第$i$个输出元素定义为：

$$
y_i=\sum_{j=0}^{i}x_j,\quad 0\le i<n
$$

CPU参考实现使用单个累加变量按顺序计算参考结果。NPU实现采用两阶段计算方式：第一阶段在每个数据块内部计算局部前缀和，并生成每个数据块的块和；第二阶段根据块间偏移量修正局部前缀和，得到全局前缀和结果。

### 3.2 数据布局

本实验输入和输出均采用一维连续存储。设数据块长度为$B$，块编号为$b$，块内元素编号为$i$，则全局下标为：

$$
idx=b\times B+i
$$

对应的输入元素和输出元素分别为：

$$
x_{b,i}=x[b\times B+i]
$$

$$
y_{b,i}=y[b\times B+i]
$$

代码中将一维数组逻辑划分为二维形式：

$$
[numBlocks,\;blockLen]
$$

其中：

$$
numBlocks=\frac{n}{blockLen}
$$

除输入数组`x`和输出数组`y`外，实验还使用两个辅助数组：

- `blockSum`：长度为`numBlocks`，保存每个数据块的块内总和。
- `blockOffset`：长度为`numBlocks`，保存每个数据块需要回加的块间偏移量。

第$b$个数据块的块和为：

$$
blockSum_b=\sum_{i=0}^{B-1}x_{b,i}
$$

第$b$个数据块的块间偏移量为：

$$
blockOffset_b=\sum_{t=0}^{b-1}blockSum_t
$$

因此，块内局部前缀和修正后的全局结果为：

$$
y_{b,i}=localScan_{b,i}+blockOffset_b
$$

### 3.3 分块策略

本实验固定数据块长度为1024。该长度对应一棵叶子节点数为$2^{10}$ 的完全二叉树，树高为10，有效节点总数为：

$$
2\times 1024-1=2047
$$

工程中使用如下常量描述块长度、树高和节点数量：
```cpp
constexpr uint32_t kBalancedTreeBlockLen = 1024;
constexpr uint32_t kBalancedTreeLevels = 10;
constexpr uint32_t kBalancedTreeNodeCount = 2 * kBalancedTreeBlockLen - 1;
```

Host侧会检查输入规模是否合法。输入长度n必须大于0，并且必须能够被blockLen整除。本实验当前代码不处理尾块，因此不支持任意长度输入。
```cpp
if (n % block_len != 0) {
    throw std::invalid_argument("n must be divisible by block_len");
}
```
NPU侧第一阶段核函数也固定要求blockLen=1024。如果传入其他块长度，核函数会直接返回。因此，在本实验中，blockLen是固定实验参数，不是可自由调节的性能参数。

多核划分采用按块连续分配的方式。设内核启动任务数为launchBlockDim，则每个逻辑任务处理一段连续的数据块：

$$
blocksPerCore=\left\lceil\frac{numBlocks}{launchBlockDim}\right\rceil
$$

第coreId个逻辑任务处理的数据块范围为：

$$
[coreId\times blocksPerCore,\; \min((coreId+1)\times blocksPerCore,\; numBlocks))
$$

该划分方式实现简单，适合本实验中每个数据块计算量一致的场景。当numBlocks小于launchBlockDim时，部分逻辑任务不参与计算。

### 3.4 平衡二叉树遍历方法

块内前缀和计算使用平衡二叉树方法。每个数据块包含1024个输入元素，作为完全二叉树的叶子节点。树高为10，有效节点总数为2047。

Device侧核函数为便于静态Tensor分配，实际申请2048个float存储单元：
```cpp
constexpr uint32_t kTreeTensorCapacity = 2 * kFixedBlockLen;
```
核函数中使用两个局部Tensor：

- `bTree`：保存各层节点对应子树的局部和。
- `cTree`：保存各层节点右边界对应的前缀和。

各层节点在一维Tensor中连续存储。第level层的起始偏移量由LevelOffset函数计算：

$$
LevelOffset(level)=\sum_{h=0}^{level-1}\frac{1024}{2^h}
$$

其中第0层为叶子层，第10层为根节点层。

块内计算分为正向遍历和反向遍历两个过程。

正向遍历阶段，核函数先将当前数据块从GM搬入UB，并写入bTree的叶子层：
```cpp
DataCopy(bTree, xGm[base], kFixedBlockLen);
```
随后按层计算子树和。第$h$层第$j$个节点由上一层相邻两个节点相加得到：

$$
B_{h,j}=B_{h-1,2j}+B_{h-1,2j+1}
$$

根节点保存当前数据块的块和，随后写入blockSum。

反向遍历阶段，核函数先将根节点对应的前缀值写入cTree，再逐层向叶子层传播前缀信息。设当前层节点使用1基编号$j$，则更新规则为：

$$
C_{h,1}=B_{h,1}
$$

$$
C_{h,j}=C_{h+1,j/2},\quad j为偶数
$$

$$
C_{h,j}=C_{h+1,(j-1)/2}+B_{h,j},\quad j为奇数且j>1
$$

完成反向遍历后，cTree的叶子层即为当前数据块的局部前缀和。核函数将该结果写回输出数组y。

### 3.5 两阶段计算流程

本实验的NPU计算流程由两个Device侧核函数和一次Host侧块间偏移计算组成。

第一阶段由block_scan_stage1_balanced_tree完成。该核函数负责执行块内平衡二叉树遍历，输出块内局部前缀和和每个数据块的块和。其主要数据流为：

```text
GM输入x → UB中的bTree/cTree → GM输出y → GM输出blockSum
```

Host侧随后将blockSum拷回主机，计算每个数据块之前所有数据块的块和，得到blockOffset。

第二阶段由add_block_offset完成。该核函数读取第一阶段得到的局部前缀和结果，并将对应块的blockOffset加到块内每个元素上。其主要数据流为：

```text
GM局部结果y → UB中的yLocal → 回加blockOffset → GM全局结果y
```

因此，本实验当前实现属于Host/Device协同的两阶段并行前缀和。块内扫描和偏移回加在NPU上完成，块间偏移量计算在Host侧完成。

### 3.6 实验参数设置

本实验默认输入规模为1048576，数据块长度固定为1024，默认启动32个逻辑计算任务，预热次数为5，重复运行次数为30。输入数据由Host侧随机生成，随机种子默认为1234，数据类型为float。

In [ ]:
N = 1048576
BLOCK_LEN = 1024
BLOCK_DIM = 32
WARMUP = 5
REPEAT = 30
SEED = 1234

NUM_BLOCKS = N // BLOCK_LEN
TREE_LEVELS = 10
TREE_NODE_COUNT = 2 * BLOCK_LEN - 1
TREE_TENSOR_CAPACITY = 2 * BLOCK_LEN

print("N:", N)
print("blockLen:", BLOCK_LEN)
print("numBlocks:", NUM_BLOCKS)
print("blockDim:", BLOCK_DIM)
print("treeLevels:", TREE_LEVELS)
print("validTreeNodes:", TREE_NODE_COUNT)
print("treeTensorCapacity:", TREE_TENSOR_CAPACITY)

需要注意，BLOCK_LEN与平衡二叉树高度直接相关。本实验代码固定BLOCK_LEN=1024，因此树高固定为10。如果需要支持其他块长度，需要同步修改Host侧参数检查、CPU平衡树模拟实现、Device侧静态Tensor容量和树层数计算逻辑。

---
## 4. 核函数开发

本节实现前缀和计算中的两个Device侧核函数。第一个核函数block_scan_stage1_balanced_tree负责在每个数据块内部执行平衡二叉树遍历，得到块内局部前缀和，并输出每个数据块的块和；第二个核函数add_block_offset负责将Host侧计算得到的块间偏移量回加到各数据块的局部结果中，得到全局前缀和。

本实验采用静态Tensor编程方式。核函数中不使用动态队列结构，而是在UB中显式申请固定容量的LocalTensor，并通过DataCopy、片上计算和写回操作完成数据处理。


### 4.1 块内平衡二叉树扫描核函数

块内扫描核函数的输入为全局内存中的原始数组x，输出为局部前缀和数组y和块和数组blockSum。每个数据块长度固定为1024，对应一棵叶子节点数为1024的完全二叉树。

该核函数的主要流程如下：

1. 根据GetBlockIdx()获取当前逻辑任务编号。
2. 按numBlocks和launchBlockDim计算当前逻辑任务负责处理的数据块范围。
3. 在UB中申请两个静态LocalTensor，分别用于保存子树局部和bTree与前缀传播结果cTree。
4. 将当前数据块从GM搬入bTree的叶子层。
5. 自叶向根逐层计算子树和，并将根节点写入blockSum。
6. 自根向叶逐层传播前缀信息，得到块内局部前缀和。
7. 将cTree叶子层写回GM中的输出数组y。

In [ ]:
from pathlib import Path 

KERNEL_DIR = Path("src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel") 
KERNEL_DIR.mkdir(parents=True, exist_ok=True) 
print("Kernel directory:", KERNEL_DIR.resolve())

In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/block_scan_stage1_balanced_tree.cpp
// Kernel 1, balanced-tree version:
//   block-local inclusive scan and block_sum generation.
//
// Static Tensor version for Experiment 4, Chen Guoliang balanced-tree method:
//   - blockLen is fixed at 1024 = 2^10
//   - logical 1-D input is still treated as [numBlocks, blockLen]
//   - each block is copied from GM into UB
//   - B tree stores subtree sums, bottom-up
//   - C tree stores prefix sums to each node's right boundary, top-down
//   - C level 0 is written back as the block-local prefix sum
//
// This is the only block-local scan kernel kept in the cleaned project.

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kFixedBlockLen = 1024;
constexpr uint32_t kTreeLevels = 10;
// A full binary tree with 1024 leaves has 2047 valid nodes. Allocate 2048
// float slots to keep the static UB tensor capacity naturally aligned.
constexpr uint32_t kTreeTensorCapacity = 2 * kFixedBlockLen;

__aicore__ inline uint32_t LevelOffset(uint32_t level) {
    uint32_t offset = 0;
    for (uint32_t h = 0; h < level; ++h) {
        offset += kFixedBlockLen >> h;
    }
    return offset;
}

__aicore__ inline void GetBlockRange(uint32_t numBlocks,
                                      uint32_t launchBlockDim,
                                      uint32_t& beginBlock,
                                      uint32_t& endBlock) {
    const uint32_t coreNum = launchBlockDim == 0 ? 1 : launchBlockDim;
    const uint32_t coreId = GetBlockIdx();
    if (coreId >= coreNum) {
        beginBlock = 0;
        endBlock = 0;
        return;
    }
    const uint32_t blocksPerCore = (numBlocks + coreNum - 1) / coreNum;
    beginBlock = coreId * blocksPerCore;
    endBlock = beginBlock + blocksPerCore;
    if (endBlock > numBlocks) {
        endBlock = numBlocks;
    }
}

__aicore__ inline void BalancedTreeScanBlock(LocalTensor<float>& bTree,
                                             LocalTensor<float>& cTree) {
    for (uint32_t h = 1; h <= kTreeLevels; ++h) {
        const uint32_t curOffset = LevelOffset(h);
        const uint32_t prevOffset = LevelOffset(h - 1);
        const uint32_t nodeCount = kFixedBlockLen >> h;
        for (uint32_t j = 0; j < nodeCount; ++j) {
            const float left = bTree.GetValue(prevOffset + 2 * j);
            const float right = bTree.GetValue(prevOffset + 2 * j + 1);
            bTree.SetValue(curOffset + j, left + right);
        }
    }

    const uint32_t rootOffset = LevelOffset(kTreeLevels);
    cTree.SetValue(rootOffset, bTree.GetValue(rootOffset));

    for (int32_t h = static_cast<int32_t>(kTreeLevels) - 1; h >= 0; --h) {
        const uint32_t level = static_cast<uint32_t>(h);
        const uint32_t curOffset = LevelOffset(level);
        const uint32_t parentOffset = LevelOffset(level + 1);
        const uint32_t nodeCount = kFixedBlockLen >> level;

        for (uint32_t j0 = 0; j0 < nodeCount; ++j0) {
            const uint32_t j = j0 + 1; // 1-based textbook node index.
            if (j == 1) {
                cTree.SetValue(curOffset + j0, bTree.GetValue(curOffset + j0));
            } else if ((j & 1u) == 0u) {
                cTree.SetValue(curOffset + j0, cTree.GetValue(parentOffset + (j / 2u - 1u)));
            } else {
                const float parentPrefix = cTree.GetValue(parentOffset + ((j - 1u) / 2u - 1u));
                const float subtreeSum = bTree.GetValue(curOffset + j0);
                cTree.SetValue(curOffset + j0, parentPrefix + subtreeSum);
            }
        }
    }
}
}  // namespace

extern "C" __global__ __aicore__ void block_scan_stage1_balanced_tree(GM_ADDR x,
                                                                       GM_ADDR y,
                                                                       GM_ADDR blockSum,
                                                                       uint32_t totalLength,
                                                                       uint32_t numBlocks,
                                                                       uint32_t blockLen,
                                                                       uint32_t launchBlockDim) {
    InitSocState();

    GlobalTensor<float> xGm;
    GlobalTensor<float> yGm;
    GlobalTensor<float> blockSumGm;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), totalLength);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), totalLength);
    blockSumGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(blockSum), numBlocks);

    if (blockLen != kFixedBlockLen) {
        return;
    }

    uint32_t beginBlock = 0;
    uint32_t endBlock = 0;
    GetBlockRange(numBlocks, launchBlockDim, beginBlock, endBlock);
    if (beginBlock >= endBlock) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> bTree = ubAllocator.Alloc<float, kTreeTensorCapacity>();
    LocalTensor<float> cTree = ubAllocator.Alloc<float, kTreeTensorCapacity>();
    bTree.SetSize(kTreeTensorCapacity);
    cTree.SetSize(kTreeTensorCapacity);

    const uint32_t rootOffset = LevelOffset(kTreeLevels);
    for (uint32_t blockId = beginBlock; blockId < endBlock; ++blockId) {
        const uint32_t base = blockId * kFixedBlockLen;

        DataCopy(bTree, xGm[base], kFixedBlockLen);
        PipeBarrier<PIPE_ALL>();

        BalancedTreeScanBlock(bTree, cTree);
        PipeBarrier<PIPE_ALL>();

        DataCopy(yGm[base], cTree, kFixedBlockLen);
        PipeBarrier<PIPE_ALL>();

        blockSumGm.SetValue(blockId, bTree.GetValue(rootOffset));
    }
}


### 4.2 块间偏移回加核函数

第一阶段核函数只能得到每个数据块内部的局部前缀和。对于第`b`个数据块，其块内结果还需要加上前面所有数据块的块和，才能得到全局前缀和。因此，Host侧会根据`blockSum`计算`blockOffset`，再由第二个核函数完成偏移量回加。

`add_block_offset`核函数的主要流程如下：

1. 根据当前AI Core编号计算负责处理的数据块范围。
2. 在UB中申请静态`LocalTensor`，用于暂存一个数据块的局部前缀和。
3. 将当前数据块的局部前缀和从GM搬入UB。
4. 读取当前数据块对应的`blockOffset`。
5. 使用`Adds`将偏移量加到块内所有元素上。
6. 将修正后的结果写回GM中的输出数组`y`。


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/add_block_offset.cpp
// Kernel 2: add block_offset to each block-local scan result.
//
// Static Tensor version for Experiment 4:
//   - no TPipe/TQue/TBuf
//   - static UB tensor capacity is fixed at kStaticMaxBlockLen
//   - GM y_block -> static UB LocalTensor -> Adds(offset) -> GM y_block

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kStaticMaxBlockLen = 8192;

__aicore__ inline void GetBlockRange(uint32_t numBlocks,
                                      uint32_t launchBlockDim,
                                      uint32_t& beginBlock,
                                      uint32_t& endBlock) {
    const uint32_t coreNum = launchBlockDim == 0 ? 1 : launchBlockDim;
    const uint32_t coreId = GetBlockIdx();
    if (coreId >= coreNum) {
        beginBlock = 0;
        endBlock = 0;
        return;
    }
    const uint32_t blocksPerCore = (numBlocks + coreNum - 1) / coreNum;
    beginBlock = coreId * blocksPerCore;
    endBlock = beginBlock + blocksPerCore;
    if (endBlock > numBlocks) {
        endBlock = numBlocks;
    }
}
}  // namespace

extern "C" __global__ __aicore__ void add_block_offset(GM_ADDR y,
                                                        GM_ADDR blockOffset,
                                                        uint32_t totalLength,
                                                        uint32_t numBlocks,
                                                        uint32_t blockLen,
                                                        uint32_t launchBlockDim) {
    InitSocState();

    GlobalTensor<float> yGm;
    GlobalTensor<float> offsetGm;
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), totalLength);
    offsetGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(blockOffset), numBlocks);

    if (blockLen == 0 || blockLen > kStaticMaxBlockLen) {
        return;
    }

    uint32_t beginBlock = 0;
    uint32_t endBlock = 0;
    GetBlockRange(numBlocks, launchBlockDim, beginBlock, endBlock);
    if (beginBlock >= endBlock) {
        return;
    }

    // Static Tensor allocation. Active length is runtime blockLen, capacity is
    // compile-time kStaticMaxBlockLen.
    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> yLocal = ubAllocator.Alloc<float, kStaticMaxBlockLen>();
    yLocal.SetSize(kStaticMaxBlockLen);

    for (uint32_t blockId = beginBlock; blockId < endBlock; ++blockId) {
        const uint32_t base = blockId * blockLen;
        const float offset = offsetGm.GetValue(blockId);

        DataCopy(yLocal, yGm[base], blockLen);
        PipeBarrier<PIPE_ALL>();

        Adds(yLocal, yLocal, offset, blockLen);
        PipeBarrier<PIPE_ALL>();

        DataCopy(yGm[base], yLocal, blockLen);
        PipeBarrier<PIPE_ALL>();
    }
}


### 4.3 核函数实现要点

本实验的两个核函数均采用按块划分的并行方式。每个逻辑任务处理若干连续数据块，任务由运行时调度至可用的AI Core，块内计算在UB中完成。由于每个数据块长度固定为`1024`，第一阶段核函数可以使用固定容量的静态Tensor表示平衡二叉树，避免动态内存管理带来的额外开销。

块内扫描核函数中，`bTree`和`cTree`的作用不同。`bTree`用于保存自叶向根遍历得到的子树局部和，根节点即当前数据块的块和。`cTree`用于保存自根向叶传播得到的前缀信息，叶子层即当前数据块的局部前缀和。该设计将前缀和中的顺序依赖转化为树形层级依赖，使块内计算具有清晰的分层结构。

偏移回加核函数中，`blockOffset`由Host侧提前计算。Device侧只负责将对应偏移量加到块内所有元素上。该阶段的计算逻辑较简单，主要体现GM与UB之间的数据搬运，以及向量加法接口`Adds`的使用。

需要注意，第一阶段核函数中的`blockLen`虽然作为参数传入，但当前实现只支持`1024`。第二阶段核函数允许`blockLen`不超过`8192`，但在本实验流程中实际仍使用`1024`，以与第一阶段保持一致。


In [ ]:
from pathlib import Path

kernel_files = [
    Path("src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/block_scan_stage1_balanced_tree.cpp"),
    Path("src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/add_block_offset.cpp"),
]

for file in kernel_files:
    print(file)
    print("exists:", file.exists(), "size:", file.stat().st_size if file.exists() else 0)


完成Device侧核函数写入后，检查两个核函数文件是否已经生成。若以下文件均存在，则说明Device侧核心代码已经写入成功。

In [ ]:
from pathlib import Path

required_files = [
    "src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/block_scan_stage1_balanced_tree.cpp",
    "src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/add_block_offset.cpp",
]

for file in required_files:
    path = Path(file)
    print(f"{file}: {'OK' if path.exists() else 'MISSING'}")


---
## 5. 结果验证与性能分析

核函数开发完成后，本节按照数据准备、工程构建、算子运行、结果验证与性能分析五个环节组织实验。Host侧负责生成输入数据和CPU串行参考结果，完成Device内存管理、核函数启动、结果回拷和指标统计；Device侧依次执行块内平衡二叉树计算和块间前缀和累加，得到全局前缀和。

本节按照完整工程流程写入CMake构建配置和运行脚本，并分别给出CPU版本与NPU版本的运行命令。运行本Notebook后，`src/02.00_intra_prefix_sum_balanced_tree`目录下将具备可直接编译运行的完整工程文件。

### 5.1 数据准备

Host侧根据输入长度`n`和随机种子生成一维单精度浮点数组，并使用同一输入分别执行CPU与NPU计算。CPU串行实现按照数组下标顺序逐项累加，生成参考结果`ref`；CPU平衡树模拟实现用于验证数据分块、平衡二叉树遍历和块间前缀和累加逻辑。NPU运行完成后，Host侧将全局前缀和结果`y`从Device侧回拷，并与`ref`逐元素比较。

本实验默认输入长度为`1048576`，数据块长度固定为`1024`，共划分为`1024`个数据块；默认启动`32`个逻辑计算任务，预热`5`次，重复运行`30`次，随机种子为`1234`。在构建工程前，先检查Notebook已经写入的关键源码文件。若以下文件均存在，说明本Notebook具备从源码写入到编译运行的基本工程结构。

In [ ]:
from pathlib import Path

required_files = [
    "src/02.00_intra_prefix_sum_balanced_tree/include/prefix_sum_common.h",
    "src/02.00_intra_prefix_sum_balanced_tree/include/prefix_sum_cpu.h",
    "src/02.00_intra_prefix_sum_balanced_tree/include/prefix_sum_balanced_tree_cpu.h",
    "src/02.00_intra_prefix_sum_balanced_tree/src/prefix_sum_cpu.cpp",
    "src/02.00_intra_prefix_sum_balanced_tree/src/prefix_sum_balanced_tree_cpu.cpp",
    "src/02.00_intra_prefix_sum_balanced_tree/src/main_balanced_tree.cpp",
    "src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/block_scan_stage1_balanced_tree.cpp",
    "src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/op_kernel/add_block_offset.cpp",
    "src/02.00_intra_prefix_sum_balanced_tree/ascend_ops/host_launch/prefix_sum_npu_main_balanced_tree.cpp",
]

for file in required_files:
    path = Path(file)
    print(f"{file}: {'OK' if path.exists() else 'MISSING'}")

### 5.2 工程构建

`CMakeLists.txt`用于组织CPU参考程序、Device侧Ascend C核函数和Host侧调用程序的编译。CPU版本不依赖NPU，用于验证输入生成、CPU串行参考结果和两阶段平衡树计算逻辑；NPU版本需要引入CANN提供的Ascend C编译配置，并链接ACL运行库。

运行NPU构建脚本时，脚本会自动识别CANN安装路径和昇腾处理器型号，将检测结果传递给构建系统，再完成核函数库与Host侧可执行程序的配置和编译。若自动检测失败，应通过命令行参数显式指定CANN安装路径和`SOC_VERSION`。

In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ascendc_balanced_tree_prefix_sum LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build Ascend C NPU demo" OFF)

add_library(prefix_sum_cpu_base
    src/prefix_sum_cpu.cpp
)
target_include_directories(prefix_sum_cpu_base PUBLIC include)
target_compile_options(prefix_sum_cpu_base PRIVATE -Wall -Wextra -Wpedantic)

add_library(prefix_sum_balanced_tree_cpu
    src/prefix_sum_balanced_tree_cpu.cpp
)
target_include_directories(prefix_sum_balanced_tree_cpu PUBLIC include)
target_link_libraries(prefix_sum_balanced_tree_cpu PUBLIC prefix_sum_cpu_base)
target_compile_options(prefix_sum_balanced_tree_cpu PRIVATE -Wall -Wextra -Wpedantic)

add_executable(prefix_sum_balanced_tree_demo src/main_balanced_tree.cpp)
target_link_libraries(prefix_sum_balanced_tree_demo PRIVATE prefix_sum_balanced_tree_cpu)
target_compile_options(prefix_sum_balanced_tree_demo PRIVATE -Wall -Wextra -Wpedantic)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode: npu/cpu/sim")
  set(SOC_VERSION "ascend910b3" CACHE STRING "Ascend SOC version, e.g. ascend910b1/ascend910b2/ascend910b3/ascend310p3")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH)
    set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation path" FORCE)
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. Check ASCEND_CANN_PATH/ASCEND_INSTALL_PATH.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(prefix_sum_balanced_tree_kernels STATIC
      ascend_ops/op_kernel/block_scan_stage1_balanced_tree.cpp
      ascend_ops/op_kernel/add_block_offset.cpp
  )
  ascendc_compile_definitions(prefix_sum_balanced_tree_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  add_executable(prefix_sum_balanced_tree_ascend_demo
      ascend_ops/host_launch/prefix_sum_npu_main_balanced_tree.cpp
  )
  target_include_directories(prefix_sum_balanced_tree_ascend_demo PRIVATE
      include
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${CMAKE_INSTALL_PREFIX}/include/prefix_sum_balanced_tree_kernels
      ${CMAKE_BINARY_DIR}/out/include/prefix_sum_balanced_tree_kernels
  )
  target_link_directories(prefix_sum_balanced_tree_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
  )
  target_link_libraries(prefix_sum_balanced_tree_ascend_demo PRIVATE
      prefix_sum_balanced_tree_kernels
      prefix_sum_balanced_tree_cpu
      ascendcl
  )
  add_dependencies(prefix_sum_balanced_tree_ascend_demo prefix_sum_balanced_tree_kernels)
endif()

install(TARGETS prefix_sum_balanced_tree_demo RUNTIME DESTINATION bin)


### 5.3 算子运行

下面写入CPU版本和NPU版本的运行脚本。CPU脚本负责编译并运行Host侧参考程序与平衡树模拟程序；NPU脚本负责加载CANN环境、配置CMake、编译Ascend C核函数和Host侧程序，并运行两阶段并行前缀和算子。

两个版本均支持设置输入长度、随机种子、预热次数和重复次数；NPU版本还支持设置Device编号和逻辑任务数。为保证结果与性能数据具有可比性，CPU与NPU运行时应使用相同的输入长度、随机种子、预热次数和重复次数。

In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/scripts/run_cpu_balanced_tree_demo.sh
#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "$0")/.." && pwd)"
BUILD_DIR="${ROOT}/build_balanced_tree_cpu"

cmake -S "${ROOT}" -B "${BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release
cmake --build "${BUILD_DIR}" -j
"${BUILD_DIR}/prefix_sum_balanced_tree_demo" \
  --n 1048576 \
  --warmup 5 \
  --repeat 30 \
  "$@"


In [ ]:
%%writefile src/02.00_intra_prefix_sum_balanced_tree/scripts/run_ascend_balanced_tree.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_balanced_tree_ascend"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"

is_cann_install_path() {
  local path="$1"
  [[ -f "${path}/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ]]
}

detect_cann_install_path() {
  local arch candidate
  arch=$(uname -m)

  if [[ -n "${ASCEND_HOME_PATH:-}" ]]; then
    candidate="${ASCEND_HOME_PATH}/${arch}-linux"
    if is_cann_install_path "${candidate}"; then
      echo "${candidate}"
      return 0
    fi
  fi

  for candidate in \
    /opt/conda/Ascend/cann-*/"${arch}-linux" \
    /usr/local/Ascend/ascend-toolkit/latest \
    /usr/local/Ascend/ascend-toolkit/*/"${arch}-linux"; do
    if is_cann_install_path "${candidate}"; then
      echo "${candidate}"
      return 0
    fi
  done
  return 1
}

ASCEND_PATH_AUTO_DETECTED=0
if [[ -n "${ASCEND_INSTALL_PATH:-}" ]] && is_cann_install_path "${ASCEND_INSTALL_PATH}"; then
  ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH}"
elif detected_path=$(detect_cann_install_path); then
  ASCEND_INSTALL_PATH="${detected_path}"
  ASCEND_PATH_AUTO_DETECTED=1
else
  ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH_DEFAULT}"
fi
SOC_VERSION_FROM_ENV="${SOC_VERSION:-}"
SOC_VERSION=""
SOC_VERSION_EXPLICIT=0
SOC_VERSION_AUTO_DETECTED=0
DEVICE_ID=0
RUN_MODE="npu"
BUILD_TYPE="Release"
CLEAN=0
WARMUP=5
REPEAT=30
N=1048576
BLOCK_DIM=32
EXTRA_ARGS=()

detect_soc_version() {
  local raw_name normalized
  if ! command -v npu-smi >/dev/null 2>&1; then
    return 1
  fi

  raw_name=$(npu-smi info 2>/dev/null \
    | sed -nE 's/^\|[[:space:]]*[0-9]+[[:space:]]+([^|]+)\|.*/\1/p' \
    | head -n 1 || true)
  normalized=$(echo "${raw_name}" \
    | tr '[:upper:]' '[:lower:]' \
    | tr -cd '[:alnum:]')

  case "${normalized}" in
    ascend910*|ascend310*) echo "${normalized}" ;;
    910*|310*) echo "ascend${normalized}" ;;
    *) return 1 ;;
  esac
}

usage() {
  cat <<USAGE
Usage: bash scripts/run_ascend_balanced_tree.sh [options] [-- extra_args_for_binary]

Options:
  -a <path>   ASCEND_INSTALL_PATH. Auto-detects common CANN installations.
  -v <soc>    SOC_VERSION. Auto-detects it from npu-smi when omitted.
  -d <id>     device id, default: 0
  -n <num>    total element count N, default: 1048576
  -b <num>    AI Core launch blockDim, default: 32
  -w <num>    warmup count, default: 5
  -r <num>    repeat count, default: 30
  -m <mode>   CMake run mode, default: npu. Keep npu for Ascend execution.
  -t <type>   CMake build type, default: Release
  -c          clean build directory before building
  -h          show help

Notes:
  balanced-tree blockLen is fixed at 1024.

Examples:
  bash scripts/run_ascend_balanced_tree.sh
  bash scripts/run_ascend_balanced_tree.sh -a /usr/local/Ascend/ascend-toolkit/latest -v ascend910b3 -d 0
  bash scripts/run_ascend_balanced_tree.sh -n 16777216 -b 32 -w 5 -r 30
  bash scripts/run_ascend_balanced_tree.sh -- --print-output
USAGE
}

while getopts ":a:v:d:n:b:w:r:m:t:ch" opt; do
  case ${opt} in
    a)
      if [[ -n "${OPTARG}" ]]; then
        ASCEND_INSTALL_PATH="${OPTARG}"
        ASCEND_PATH_AUTO_DETECTED=0
      fi
      ;;
    v)
      SOC_VERSION="${OPTARG}"
      SOC_VERSION_EXPLICIT=1
      ;;
    d) DEVICE_ID="${OPTARG}" ;;
    n) N="${OPTARG}" ;;
    b) BLOCK_DIM="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    m) RUN_MODE="${OPTARG}" ;;
    t) BUILD_TYPE="${OPTARG}" ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))

if [[ $# -gt 0 && "$1" == "--" ]]; then
  shift
fi
EXTRA_ARGS=("$@")

if [[ "${SOC_VERSION_EXPLICIT}" == "0" ]]; then
  if detected_soc=$(detect_soc_version); then
    SOC_VERSION="${detected_soc}"
    SOC_VERSION_AUTO_DETECTED=1
  elif [[ -n "${SOC_VERSION_FROM_ENV}" ]]; then
    SOC_VERSION="${SOC_VERSION_FROM_ENV}"
  else
    echo "Cannot detect SOC_VERSION from npu-smi. Use -v <soc>." >&2
    exit 1
  fi
fi

if [[ ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  echo "ASCEND_INSTALL_PATH does not exist: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

if [[ "${ASCEND_PATH_AUTO_DETECTED}" == "1" ]]; then
  echo "[INFO] Auto-detected ASCEND_INSTALL_PATH=${ASCEND_INSTALL_PATH}"
fi
if [[ "${SOC_VERSION_AUTO_DETECTED}" == "1" ]]; then
  echo "[INFO] Auto-detected SOC_VERSION=${SOC_VERSION}"
fi
echo "[INFO] ACL logical device=${DEVICE_ID}, blockDim=${BLOCK_DIM}, warmup=${WARMUP}, repeat=${REPEAT}"

if [[ "${CLEAN}" == "1" ]]; then
  rm -rf "${BUILD_DIR}"
fi
mkdir -p "${BUILD_DIR}"

if [[ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  # shellcheck disable=SC1090
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
fi

# Some CANN packages put ascendc.cmake, Ascend C headers, and Bisheng directly
# under tikcpp/, asc/, and ccec_compiler/.  Their CMake files may still look
# for those components below ascendc_devkit/.  Build a compatibility view
# locally instead of modifying the system CANN installation.
ASCEND_CMAKE_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
ASCENDC_CMAKE_NEW_LAYOUT="${ASCEND_INSTALL_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
BISHENG_REAL="${ASCEND_INSTALL_PATH}/ccec_compiler/bin/bisheng"
BISHENG_EXPECTED="${ASCEND_INSTALL_PATH}/ascendc_devkit/ccec_compiler/bin/bisheng"

if [[ -f "${ASCENDC_CMAKE_NEW_LAYOUT}" && -x "${BISHENG_REAL}" && ! -x "${BISHENG_EXPECTED}" ]]; then
  ASCEND_CMAKE_PACKAGE_PATH="${BUILD_DIR}/cann_package_compat"
  mkdir -p "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit"

  for entry in "${ASCEND_INSTALL_PATH}"/*; do
    [[ -e "${entry}" ]] || continue
    name=$(basename "${entry}")
    [[ "${name}" == "ascendc_devkit" ]] && continue
    ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/${name}"
  done

  if [[ -d "${ASCEND_INSTALL_PATH}/ascendc_devkit" ]]; then
    for entry in "${ASCEND_INSTALL_PATH}/ascendc_devkit"/*; do
      [[ -e "${entry}" ]] || continue
      name=$(basename "${entry}")
      [[ "${name}" == "ccec_compiler" ]] && continue
      ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${name}"
    done
  fi

  for component in asc tikcpp ccec_compiler; do
    if [[ -e "${ASCEND_INSTALL_PATH}/${component}" ]]; then
      ln -sfn "${ASCEND_INSTALL_PATH}/${component}" \
        "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${component}"
    fi
  done
  echo "[INFO] Using CANN package compatibility view: ${ASCEND_CMAKE_PACKAGE_PATH}"
fi

export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PACKAGE_PATH}"
export SOC_VERSION

cd "${BUILD_DIR}"

cmake "${SCRIPT_DIR}" \
  -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${ASCEND_CMAKE_PACKAGE_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PACKAGE_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE="${RUN_MODE}"

cmake --build . -j

BIN="${BUILD_DIR}/prefix_sum_balanced_tree_ascend_demo"
if [[ ! -x "${BIN}" ]]; then
  echo "Cannot find executable: ${BIN}" >&2
  exit 1
fi

CMD=("${BIN}" --device "${DEVICE_ID}" --n "${N}" --block-dim "${BLOCK_DIM}" --warmup "${WARMUP}" --repeat "${REPEAT}")
CMD+=("${EXTRA_ARGS[@]}")

echo "[RUN] ${CMD[*]}"
"${CMD[@]}"


In [ ]:
!chmod +x src/02.00_intra_prefix_sum_balanced_tree/scripts/run_cpu_balanced_tree_demo.sh
!chmod +x src/02.00_intra_prefix_sum_balanced_tree/scripts/run_ascend_balanced_tree.sh
!find src/02.00_intra_prefix_sum_balanced_tree -maxdepth 3 -type f | sort

#### 5.3.1 运行CPU版本

先运行CPU版本，确认CPU串行参考结果与两阶段平衡树模拟结果一致。该命令会完成CPU工程构建和运行，并将输出保存到`results/prefix_sum_cpu_result.txt`。CPU程序同时生成串行参考结果和两阶段平衡树模拟结果，可用于验证数据分块、正向遍历、反向遍历以及块间前缀和累加逻辑。

In [ ]:
!cd src/02.00_intra_prefix_sum_balanced_tree && bash scripts/run_cpu_balanced_tree_demo.sh --n 1048576 | tee results/prefix_sum_cpu_result.txt

#### 5.3.2 运行NPU版本

在已安装CANN且存在NPU的环境中，执行下面命令运行NPU版本。默认输入长度为`1048576`，数据块长度固定为`1024`，共包含`1024`个数据块，并启动`32`个逻辑计算任务。各任务分别处理连续的数据块范围；在默认配置下，每个任务依次处理`32`个数据块。

该命令覆盖第一阶段块内平衡二叉树计算与块和生成、Host侧块和回拷及排他前缀和计算、第二阶段块间前缀和累加，以及最终结果回拷和误差统计。运行结果保存到`results/prefix_sum_npu_result.txt`。

In [ ]:
# 在已安装CANN且存在NPU的环境中执行；脚本自动检测CANN路径和昇腾处理器型号
!cd src/02.00_intra_prefix_sum_balanced_tree && bash scripts/run_ascend_balanced_tree.sh | tee results/prefix_sum_npu_result.txt


### 5.4 结果验证

通过分析`max_abs`和`max_rel`实验指标进行结果验证，其中，`max_abs`表示计算结果与CPU串行参考结果之间的最大绝对误差，`max_rel`表示最大相对误差。具体计算公式如下：

$$
max\_abs=\max_i|y_i-ref_i|
$$

$$
max\_rel=\max_i\frac{|y_i-ref_i|}{\max(|ref_i|,10^{-12})}
$$

其中，$y_i$为CPU平衡树模拟结果或NPU并行结果，$ref_i$为CPU串行参考结果。前缀和属于浮点累加计算，CPU串行实现按照数组顺序逐项累加，平衡树实现按照树形结构改变了加法结合顺序，因此可能出现微小数值差异。

进行结果验证时，应首先确认CPU与NPU使用相同的输入长度和随机种子，并确认输入长度能够被`1024`整除。若`max_abs`与`max_rel`处于较小范围内，同时样例输出中的`y`与`ref`基本一致，可以认为当前计算结果可信。若误差明显增大，应依次检查第一阶段生成的`blockSum`、Host侧计算的`blockOffset`以及第二阶段块间前缀和累加是否与数据块编号正确对应。

In [ ]:
from pathlib import Path

for path in [
    Path("src/02.00_intra_prefix_sum_balanced_tree/results/prefix_sum_cpu_result.txt"),
    Path("src/02.00_intra_prefix_sum_balanced_tree/results/prefix_sum_npu_result.txt"),
]:
    print("=", path)
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="ignore")[:4000])
    else:
        print("not found")

### 5.5 性能分析

通过分析`scan_us`、`offset_us`、`add_us`、`total_us`和`GB/s`实验指标进行性能分析。其中，`scan_us`表示第一阶段块内平衡二叉树计算时间，`add_us`表示第二阶段块间前缀和累加时间，`total_us`表示各阶段的整体执行时间，`GB/s`表示根据数据访问量和总执行时间估算的有效带宽。对于NPU版本，`offset_us`包括块和从Device侧回拷、Host侧排他前缀和计算以及块间累加值传回Device侧的时间；对于CPU模拟版本，`offset_us`仅表示CPU侧块间累加值的计算时间。比较分阶段耗时时，应注意两种版本的`offset_us`统计范围不同。

本实验默认输入数据长度为`1048576`，数据块长度为`1024`，AI Core个数为`32`，随机种子为`1234`。实验中设置计时前预热`5`次、重复运行`30`次，并取实验结果的平均值，以降低实验误差的影响。

性能分析时，定义加速比S，表示CPU侧基线总时间与NPU总执行时间的比值，公式为：

$$
S=\frac{T_{CPU}}{T_{NPU}}
$$

其中，当$S>1$时，表示当前配置下NPU并行实现的总执行时间低于CPU侧基线；当$S<1$时，则表示CPU侧基线的总执行时间更低。分析时还应结合`GB/s`判断有效带宽是否得到提升，避免仅依据单一计时指标评价整体性能。

NPU各阶段的耗时占比可以按下式计算：

$$
P_{scan}=\frac{scan\_us}{total\_us}\times100\%,\quad
P_{offset}=\frac{offset\_us}{total\_us}\times100\%,\quad
P_{add}=\frac{add\_us}{total\_us}\times100\%
$$

占比最高的阶段通常是当前实现的主要性能开销。若`scan_us`占比较高，应重点分析平衡二叉树正向遍历、反向遍历、逐节点访问、同步和数据搬运；若`offset_us`占比较高，应检查Host与Device之间的数据传输及Host侧排他前缀和计算；若`add_us`占比较高，应分析第二阶段的数据搬入、向量加法和结果写回过程。

---
## 6. 实验总结

本实验按照环境准备、算子分析、核函数开发和正确性验证四个阶段，实现了基于Ascend C静态Tensor的并行前缀和计算。

* 算子分析明确了前缀和的输入输出、计算公式、连续数据布局、分块策略和平衡二叉树遍历方法。
* 块内扫描核函数在UB中构造平衡二叉树结构，通过正向遍历生成子树局部和，通过反向遍历传播前缀信息，得到每个数据块的局部前缀和。
* 块间偏移回加核函数根据Host侧计算得到的块间偏移量，对各数据块的局部结果进行修正，形成全局前缀和。
* Host侧程序完成输入数据生成、CPU参考结果计算、Device内存管理、核函数启动、结果回拷和误差统计。
* 正确性验证通过最大绝对误差和最大相对误差对比Host侧参考结果与Device侧计算结果，判断并行计算结果是否可信。

通过本实验，可以理解平衡二叉树结构在并行前缀和中的依赖组织方式，掌握数据分块、块内扫描和块间偏移修正在并行算法设计中的作用，并熟悉Ascend C静态Tensor编程中数据搬运、片上计算和Host/Device协同执行的基本流程。


